# Esperimento 01 — dinamica latente input-only: GRU vs CfC

Primo confronto sul micro-neurone Hay a quattro compartimenti. Entrambe le reti ricevono **soltanto spike presinaptici**. Il burn-in contiene soltanto spike e inizializza lo stato ricorrente; tensioni, gate, calcio e conduttanze sono esclusivamente target.

CfC è il layer pubblicato della libreria `ncps`, non una variante costruita ad hoc.

In [ ]:
from pathlib import Path
import importlib.util, os, subprocess, sys

REPOSITORY = 'https://github.com/Zagred47/LearningSingleCompartiment.git'
ROOT = Path('/kaggle/working/LearningSingleCompartiment')
if (ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if importlib.util.find_spec('ncps') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ncps==1.0.1'], check=True)
print('Project root:', ROOT)

In [ ]:
import copy, json, math, random, time
from dataclasses import asdict, dataclass

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from tqdm.auto import tqdm

from hay_single_compartment import (
    InputOnlyCfC, InputOnlyGRU, count_trainable_parameters,
    validate_micro_dataset,
)

SEED = 20260730
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'none')

In [ ]:
mounted = sorted(Path('/kaggle/input').glob('**/hay_micro_4c_v1.h5'))
local = Path('/kaggle/working/hay_micro_4c_v1.h5')
DATASET_PATH = mounted[0] if mounted else local
if not DATASET_PATH.is_file():
    raise FileNotFoundError('Dataset non trovato: esegui prima kaggle_micro_hay_4comp_dataset.ipynb')
report = validate_micro_dataset(DATASET_PATH)
if not report['valid']:
    raise ValueError(f'Dataset incompatibile: {report["issues"]}. Rigeneralo col notebook aggiornato.')
with h5py.File(DATASET_PATH, 'r') as h5:
    state_names = json.loads(h5.attrs['state_names_json'])
    input_names = json.loads(h5.attrs['input_names_json'])
    dataset_config = json.loads(h5.attrs['config_json'])
print(DATASET_PATH, '| states:', len(state_names), '| raw spike channels:', len(input_names))

## Risoluzione del learner

Il teacher resta a 0,1 ms. Per rendere pratico il training ricorrente, cinque micro-bin consecutivi vengono **concatenati**, non sommati: ogni passo di rete copre 0,5 ms ma conserva quale sinapsi ha scaricato in ciascuno dei cinque istanti. Nessun evento viene perso e nessuno stato fisico entra nell'input.

In [ ]:
@dataclass(frozen=True)
class TrainConfig:
    temporal_bin: int = 5
    batch_trajectories: int = 10
    chunk_steps: int = 256
    epochs: int = 30
    patience: int = 7
    learning_rate: float = 5e-4
    weight_decay: float = 1e-5
    gradient_clip: float = 1.0
    cfc_hidden: int = 192
    cfc_embedding: int = 128
    cfc_backbone: int = 192
    cfc_backbone_layers: int = 2

CFG = TrainConfig()
RAW_DT_MS = float(dataset_config['dt_ms'])
MODEL_DT_MS = RAW_DT_MS * CFG.temporal_bin
RESULTS = Path('/kaggle/working/hay_micro_input_only_01')
CHECKPOINTS = RESULTS / 'checkpoints'
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
print(asdict(CFG), '| model dt:', MODEL_DT_MS, 'ms')

In [ ]:
def pack_spikes(values, factor):
    usable = values.shape[1] // factor * factor
    values = values[:, :usable]
    return values.reshape(values.shape[0], usable // factor, factor * values.shape[2]).astype(np.float32)

def load_split(name):
    with h5py.File(DATASET_PATH, 'r') as h5:
        burnin = pack_spikes(h5[f'{name}/burnin_inputs'][...], CFG.temporal_bin)
        burnin_states = h5[f'{name}/burnin_states'][:, ::CFG.temporal_bin, :].astype(np.float32)
        inputs = pack_spikes(h5[f'{name}/inputs'][...], CFG.temporal_bin)
        states = h5[f'{name}/states'][:, ::CFG.temporal_bin, :].astype(np.float32)
    expected = inputs.shape[1] + 1
    states = states[:, :expected]
    burnin_states = burnin_states[:, :burnin.shape[1]+1]
    return burnin, burnin_states, inputs, states

train_burnin, train_burnin_y, train_x, train_y = load_split('train')
val_burnin, val_burnin_y, val_x, val_y = load_split('validation')
test_burnin, test_burnin_y, test_x, test_y = load_split('test')
normalization_source = np.concatenate((train_burnin_y, train_y[:, 1:]), axis=1)
state_mean = normalization_source.reshape(-1, normalization_source.shape[-1]).mean(0)
state_std = normalization_source.reshape(-1, normalization_source.shape[-1]).std(0)
state_std = np.maximum(state_std, 1e-6)
train_burnin_y_n = (train_burnin_y - state_mean) / state_std
train_y_n = (train_y - state_mean) / state_std
val_y_n = (val_y - state_mean) / state_std
test_y_n = (test_y - state_mean) / state_std
print('train:', train_burnin.shape, train_burnin_y.shape, train_x.shape, train_y.shape)
print('validation:', val_burnin.shape, val_x.shape, val_y.shape)

In [ ]:
INPUT_DIM, STATE_DIM = train_x.shape[-1], train_y.shape[-1]
cfc_probe = InputOnlyCfC(
    INPUT_DIM, STATE_DIM, hidden_dim=CFG.cfc_hidden,
    input_embedding_dim=CFG.cfc_embedding, backbone_units=CFG.cfc_backbone,
    backbone_layers=CFG.cfc_backbone_layers, decoder_dim=CFG.cfc_hidden,
)
cfc_params = count_trainable_parameters(cfc_probe)
candidates = []
for hidden in range(64, 513, 8):
    probe = InputOnlyGRU(INPUT_DIM, STATE_DIM, hidden_dim=hidden, layers=1, decoder_dim=hidden)
    candidates.append((abs(count_trainable_parameters(probe) - cfc_params), hidden, count_trainable_parameters(probe)))
_, GRU_HIDDEN, gru_params = min(candidates)
del cfc_probe
models = {
    'gru': InputOnlyGRU(INPUT_DIM, STATE_DIM, hidden_dim=GRU_HIDDEN, layers=1, decoder_dim=GRU_HIDDEN),
    'cfc': InputOnlyCfC(
        INPUT_DIM, STATE_DIM, hidden_dim=CFG.cfc_hidden, input_embedding_dim=CFG.cfc_embedding,
        backbone_units=CFG.cfc_backbone, backbone_layers=CFG.cfc_backbone_layers,
        decoder_dim=CFG.cfc_hidden,
    ),
}
parameter_table = pd.DataFrame([
    {'model': name, 'parameters': count_trainable_parameters(model)} for name, model in models.items()
]).sort_values('parameters')
display(parameter_table)
print('GRU hidden scelto automaticamente:', GRU_HIDDEN, '| scarto parametri:', abs(gru_params-cfc_params)/cfc_params)

In [ ]:
def tensor(values, indices=None):
    if indices is not None: values = values[indices]
    return torch.as_tensor(values, device=DEVICE)

def timespans(batch, steps):
    return torch.full((batch, steps), MODEL_DT_MS, device=DEVICE)

def prime_hidden(model, burnin):
    hidden = None
    with torch.no_grad():
        for start in range(0, burnin.shape[1], CFG.chunk_steps):
            chunk = burnin[:, start:start+CFG.chunk_steps]
            _, hidden = model(chunk, hidden, timespans(chunk.shape[0], chunk.shape[1]))
            hidden = model.detach_hidden(hidden)
    return hidden

@torch.no_grad()
def validation_loss(model, burnin_np, x_np, y_np):
    model.eval(); total = 0.0; elements = 0
    for offset in range(0, len(x_np), CFG.batch_trajectories):
        indices = np.arange(offset, min(offset + CFG.batch_trajectories, len(x_np)))
        burnin, x, y = tensor(burnin_np, indices), tensor(x_np, indices), tensor(y_np, indices)
        hidden = prime_hidden(model, burnin)
        initial = model.decode_hidden(hidden)
        total += torch.square(initial - y[:, 0]).sum().item(); elements += initial.numel()
        for start in range(0, x.shape[1], CFG.chunk_steps):
            chunk = x[:, start:start+CFG.chunk_steps]
            prediction, hidden = model(chunk, hidden, timespans(chunk.shape[0], chunk.shape[1]))
            target = y[:, start+1:start+1+chunk.shape[1]]
            total += torch.square(prediction - target).sum().item(); elements += target.numel()
            hidden = model.detach_hidden(hidden)
    return total / elements

def train_model(name, model):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    history, best, best_epoch, stale = [], float('inf'), 0, 0
    started = time.perf_counter()
    epoch_bar = tqdm(range(1, CFG.epochs + 1), desc=f'train {name}', position=0)
    for epoch in epoch_bar:
        model.train(); order = np.random.permutation(len(train_x)); running = 0.0; updates = 0
        for offset in range(0, len(order), CFG.batch_trajectories):
            indices = order[offset:offset+CFG.batch_trajectories]
            burnin, burnin_y = tensor(train_burnin, indices), tensor(train_burnin_y_n, indices)
            x, y = tensor(train_x, indices), tensor(train_y_n, indices)
            hidden = None
            for start in range(0, burnin.shape[1], CFG.chunk_steps):
                chunk = burnin[:, start:start+CFG.chunk_steps]
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
                    prediction, next_hidden = model(chunk, hidden, timespans(chunk.shape[0], chunk.shape[1]))
                    target = burnin_y[:, start+1:start+1+chunk.shape[1]]
                    loss = torch.mean(torch.square(prediction - target))
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip)
                scaler.step(optimizer); scaler.update()
                hidden = model.detach_hidden(next_hidden)
                running += loss.item(); updates += 1
            for start in range(0, x.shape[1], CFG.chunk_steps):
                chunk = x[:, start:start+CFG.chunk_steps]
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
                    prediction, next_hidden = model(chunk, hidden, timespans(chunk.shape[0], chunk.shape[1]))
                    target = y[:, start+1:start+1+chunk.shape[1]]
                    loss = torch.mean(torch.square(prediction - target))
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip)
                scaler.step(optimizer); scaler.update()
                hidden = model.detach_hidden(next_hidden)
                running += loss.item(); updates += 1
        scheduler.step()
        val = validation_loss(model, val_burnin, val_x, val_y_n)
        elapsed = time.perf_counter() - started
        eta = elapsed / epoch * (CFG.epochs - epoch)
        row = {'epoch': epoch, 'train_loss': running/updates, 'validation_loss': val, 'elapsed_s': elapsed, 'eta_s': eta}
        history.append(row); epoch_bar.set_postfix(train=f'{row["train_loss"]:.3e}', val=f'{val:.3e}', eta=f'{eta/60:.1f}m')
        if val < best:
            best, best_epoch, stale = val, epoch, 0
            torch.save({'model': model.state_dict(), 'config': asdict(CFG), 'state_mean': state_mean, 'state_std': state_std, 'state_names': state_names, 'input_names': input_names}, CHECKPOINTS / f'{name}.pt')
        else:
            stale += 1
            if stale >= CFG.patience:
                print(f'{name}: early stop a epoch {epoch}; best epoch {best_epoch}')
                break
    checkpoint = torch.load(CHECKPOINTS / f'{name}.pt', map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    pd.DataFrame(history).to_csv(RESULTS / f'{name}_history.csv', index=False)
    return model, history, best, best_epoch


In [ ]:
trained, histories, summaries = {}, {}, []
for name, model in models.items():
    print('\n' + '='*80 + f'\n{name}: {count_trainable_parameters(model):,} parametri')
    trained[name], histories[name], best, best_epoch = train_model(name, model)
    summaries.append({'model': name, 'parameters': count_trainable_parameters(model), 'best_epoch': best_epoch, 'validation_loss': best})

In [ ]:
@torch.no_grad()
def predict_complete(model, burnin_np, x_np):
    model.eval(); outputs = []
    for offset in tqdm(range(0, len(x_np), CFG.batch_trajectories), desc='full input-only rollout', leave=False):
        indices = np.arange(offset, min(offset + CFG.batch_trajectories, len(x_np)))
        burnin, x = tensor(burnin_np, indices), tensor(x_np, indices)
        hidden = prime_hidden(model, burnin)
        pieces = [model.decode_hidden(hidden).unsqueeze(1)]
        for start in range(0, x.shape[1], CFG.chunk_steps):
            chunk = x[:, start:start+CFG.chunk_steps]
            prediction, hidden = model(chunk, hidden, timespans(chunk.shape[0], chunk.shape[1]))
            pieces.append(prediction); hidden = model.detach_hidden(hidden)
        normalized = torch.cat(pieces, 1).float().cpu().numpy()
        outputs.append(normalized * state_std + state_mean)
    return np.concatenate(outputs, 0)

predictions = {}
voltage_indices = [state_names.index(f'{region}.v_mV') for region in ('soma','basal','trunk','tuft')]
for row in summaries:
    name = row['model']; predictions[name] = predict_complete(trained[name], test_burnin, test_x)
    error = predictions[name] - test_y
    row['test_soma_voltage_rmse_mV'] = float(np.sqrt(np.mean(error[..., voltage_indices[0]]**2)))
    row['test_all_voltage_rmse_mV'] = float(np.sqrt(np.mean(error[..., voltage_indices]**2)))
    row['test_mean_normalized_rmse'] = float(np.mean(np.sqrt(np.mean((error/state_std)**2, axis=(0,1)))))
comparison = pd.DataFrame(summaries).sort_values('validation_loss')
comparison.to_csv(RESULTS / 'comparison.csv', index=False)
display(comparison)

In [ ]:
horizons_ms = [1, 5, 10, 25, 50, 100, 200, 500, 1000, 2000, 5000]
horizon_rows = []
for name, prediction in predictions.items():
    for horizon in horizons_ms:
        index = min(int(round(horizon / MODEL_DT_MS)), test_y.shape[1] - 1)
        point_error = prediction[:, index, voltage_indices[0]] - test_y[:, index, voltage_indices[0]]
        cumulative = prediction[:, :index+1, voltage_indices[0]] - test_y[:, :index+1, voltage_indices[0]]
        horizon_rows.append({'model': name, 'horizon_ms': horizon, 'point_soma_voltage_rmse_mV': float(np.sqrt(np.mean(point_error**2))), 'cumulative_soma_voltage_rmse_mV': float(np.sqrt(np.mean(cumulative**2)))})
horizon_table = pd.DataFrame(horizon_rows)
horizon_table.to_csv(RESULTS / 'horizon_metrics.csv', index=False)
display(horizon_table.pivot(index='horizon_ms', columns='model', values='point_soma_voltage_rmse_mV'))

In [ ]:
winner = comparison.iloc[0].model
duration_to_plot_ms = min(2000, dataset_config['duration_ms'])
end = int(round(duration_to_plot_ms / MODEL_DT_MS)) + 1
time_axis = np.arange(end) * MODEL_DT_MS
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True, constrained_layout=True)
for axis, region, state_index in zip(axes, ('soma','basal','trunk','tuft'), voltage_indices):
    axis.plot(time_axis, test_y[0, :end, state_index], label='teacher', lw=1.2)
    axis.plot(time_axis, predictions[winner][0, :end, state_index], label=winner, lw=1.0, alpha=.85)
    axis.set_ylabel(region + ' V (mV)'); axis.legend()
axes[-1].set_xlabel('time (ms)')
fig.savefig(RESULTS / 'winner_voltage_rollout.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
for name, history in histories.items():
    frame = pd.DataFrame(history); ax.semilogy(frame.epoch, frame.validation_loss, label=name)
ax.set(xlabel='epoch', ylabel='validation normalized MSE', title='Training input-only'); ax.legend()
fig.savefig(RESULTS / 'training_curves.png', dpi=150)
plt.show()

In [ ]:
experiment = {
    'experiment': 'micro_hay_input_only_gru_vs_cfc_01',
    'dataset': str(DATASET_PATH), 'dataset_schema': '1.2.0',
    'network_inputs': 'binary presynaptic spikes only',
    'teacher_state_feedback': False, 'burnin_is_spike_only': True,
    'raw_dt_ms': RAW_DT_MS, 'model_dt_ms': MODEL_DT_MS,
    'temporal_encoding': f'{CFG.temporal_bin} ordered binary microbins concatenated',
    'cfc_time_unit': f'one constant recurrent step = {MODEL_DT_MS} ms',
    'training_config': asdict(CFG),
}
(RESULTS / 'experiment.json').write_text(json.dumps(experiment, indent=2), encoding='utf-8')
np.savez_compressed(RESULTS / 'test_predictions.npz', teacher=test_y, **predictions)
print('Risultati:', RESULTS, '| winner validation-selected:', winner)

In [ ]:
from shutil import make_archive
zip_path = Path(make_archive('/kaggle/working/hay_micro_input_only_01_complete', 'zip', root_dir=RESULTS.parent, base_dir=RESULTS.name))
print('ZIP pronto:', zip_path, f'({zip_path.stat().st_size/2**20:.1f} MiB)')
print('Scaricalo dalla sezione Output di Kaggle dopo Save Version.')